# Custom contact processes

A **rate** is a rule that turns "who is in which compartment" into a number on
a flow. Infection already has a packaged rule
({class}`~summer4.epi.ForceOfInfection`). This page shows how to build your own
with the same pieces — without needing to be a software developer.

We climb four rungs. Most modellers stop on rung 1 or 2.

1. **Compose with** {class}`~summer4.Reduce` — write the biology as arithmetic
   on group counts.
2. **Wrap a function with** {func}`~summer4.defer` — ordinary Python in the
   rate slot, when the formula is easier to write as code than as nodes.
3. **Customise FOI with** `kind=` — keep FOI's mixing and infectiousness knobs,
   change only the shedding formula.
4. **Appendix:** name a reusable process class (for package authors).

Every claim is asserted so the page stays a runnable test.


In [ ]:
import numpy as np

from summer4 import (
    Compartments,
    EntryFlow,
    ExitFlow,
    FlowModel,
    FlowRef,
    Property,
    PropertyMap,
    Reduce,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix


## Reminder: rates sit on flows

```python
TransitionFlow("infection", state["S"], state["I"], some_rate)
```

By default, people leave the source at **rate × how many are in the source**.
So `some_rate` is a **per-person hazard**, not a total count of events.

{class}`~summer4.EntryFlow` is different: its rate is already a total mass
entering the system (no multiply by destination size).


## Rung 1 — compose with `Reduce`

{class}`~summer4.Reduce` sums compartment sizes over a grouping property.
Optional `where=` keeps only matching compartments (for example infectious).

A frequency-dependent force of infection is exactly:

> contact rate × (infectious count in the group) / (everyone in the group)

Written with `Reduce`, that is the rate on an infection flow. We use a
singleton `pop` property so unstratified models still have a grouping axis
(the same pattern as {doc}`../summer2/01-basic-model`).


In [ ]:
state = Property("state", ("S", "I", "R"))
pop = Property("pop", ("all",))
pmap = PropertyMap.from_property(state).stratify(pop)

beta = 0.4
recovery = 0.1

hazard = beta * (
    Reduce(sum_over=pop, where=state["I"]) / Reduce(sum_over=pop)
)

m_reduce = FlowModel(pmap)
m_reduce.add_flow(TransitionFlow("infection", state["S"], state["I"], hazard))
m_reduce.add_flow(TransitionFlow("recovery", state["I"], state["R"], recovery))
cm_reduce = m_reduce.compile()

mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
m_foi = FlowModel(pmap)
m_foi.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=pop,
            mixing=mixing,
            kind=FOIKind.FREQUENCY,
            contact_rate=beta,
        ),
    )
)
m_foi.add_flow(TransitionFlow("recovery", state["I"], state["R"], recovery))
cm_foi = m_foi.compile()

y0 = np.array([990.0, 10.0, 0.0])
plan = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 40.0, 41),
)
r_reduce = cm_reduce.run({}, y0, t0=0.0, t1=40.0, dt=0.1, save=plan, solver="euler")
r_foi = cm_foi.run({}, y0, t0=0.0, t1=40.0, dt=0.1, save=plan, solver="euler")

np.testing.assert_allclose(
    np.asarray(r_reduce["comp"].values.data),
    np.asarray(r_foi["comp"].values.data),
    rtol=1e-5,
    atol=1e-5,
)
print("Reduce hazard matches ForceOfInfection(kind=FOIKind.FREQUENCY)")


### Domain variants on the same rung

| Process | Expression sketch |
|---|---|
| Density transmission | `beta * Reduce(sum_over=pop, where=state["I"])` |
| Predation on prey | `ExitFlow(prey, alpha * Reduce(..., where=predator))` |
| Birth ∝ population | `EntryFlow(prey, r * Reduce(..., where=prey))` |

Below: a tiny Lotka–Volterra split into separate flows. Predation removes prey
at hazard `α × predators`; conversion births predators from that kill mass via
{class}`~summer4.FlowRef`.


In [ ]:
species = Property("species", ("prey", "predator"))
pop_lv = Property("pop", ("all",))
pmap_lv = PropertyMap.from_property(species).stratify(pop_lv)

alpha, beta_conv, r, delta = 0.01, 0.5, 0.8, 0.4

m_lv = FlowModel(pmap_lv)
m_lv.add_flow(
    EntryFlow("prey_birth", species["prey"], r * Reduce(sum_over=pop_lv, where=species["prey"]))
)
m_lv.add_flow(
    ExitFlow(
        "predation",
        species["prey"],
        alpha * Reduce(sum_over=pop_lv, where=species["predator"]),
    )
)
m_lv.add_flow(EntryFlow("conversion", species["predator"], beta_conv * FlowRef("predation").sum()))
m_lv.add_flow(ExitFlow("pred_death", species["predator"], delta))
cm_lv = m_lv.compile()

y_lv = np.array([40.0, 9.0])
dy = np.asarray(cm_lv.vector_field(0.0, y_lv, {}))
prey, predator = y_lv
expected = np.array(
    [
        r * prey - alpha * predator * prey,
        beta_conv * alpha * predator * prey - delta * predator,
    ]
)
np.testing.assert_allclose(dy, expected, rtol=1e-5, atol=1e-5)
assert dy[0] > 0.0 and dy[1] < 0.0  # prey growing, predators declining at this point
print("Lotka–Volterra flows match classic dN/dt, dP/dt")


## Rung 2 — wrap a function with `defer`

Use this when the hazard is easier to write as ordinary Python than as rate
nodes, and it is not a force of infection. `defer(fn)(...)` builds a node.
Pass every input the function needs — time, parameters, other flows — as
arguments. `fn` is called with traced JAX values, so use `jax.numpy` inside
it. `jnp.sin(Time())` does not work; `defer` around a function that calls
`jnp.sin` does.

`name=` is optional. It asserts that two different function objects are the
same program, so a sweep that rebuilds the model can share one compiled
program. The default key is the function's identity, which is what keeps two
closures that captured different numbers from silently sharing a cache entry.


In [ ]:
import jax.numpy as jnp

from summer4 import Param, PropertyData, Time, defer

def seasonal(t, amp):
    return 0.1 * (1.0 + amp * jnp.sin(t))

rate_d = defer(seasonal)(Time(), Param("amp"))
state_d = Property("state", ("Y",))
pmap_d = PropertyMap.from_property(state_d)
model_d = FlowModel(pmap_d)
model_d.add_flow(EntryFlow("season", state_d["Y"], rate_d))
compiled_d = model_d.compile()
y_d = PropertyData.wrap(pmap_d, np.array([0.0]))
defer_got = float(np.asarray(compiled_d.vector_field(1.0, y_d, {"amp": 0.5}).data)[0])
defer_expected = 0.1 * (1.0 + 0.5 * np.sin(1.0))
np.testing.assert_allclose(defer_got, defer_expected, rtol=1e-5, atol=1e-5)
print(f"defer seasonal rate {defer_got:.6f}")


## Rung 3 — customise FOI with `kind=`

Use this when you want FOI's **mixing matrix** and **infectiousness** knobs,
but a different shedding law. Pass a small function as `kind=`:

`kind(infectious, denominator) -> GroupedRate`

Built-in `"frequency"` and `"density"` are just named versions of that idea.
Reimplementing frequency as a callable must match bit-for-bit.


In [ ]:
from summer4 import GroupedRate

age = Property("age", ("young", "old"))
pmap_age = PropertyMap.from_property(state).stratify(age)


def as_frequency(infectious: GroupedRate, denominator: GroupedRate) -> GroupedRate:
    return infectious / denominator


def build(kind: object) -> object:
    m = FlowModel(pmap_age)
    foi = ForceOfInfection(
        "infection",
        infectious=state["I"],
        group_by=age,
        kind=kind,  # type: ignore[arg-type]
        contact_rate=1.0,
        mixing=MixingMatrix(age, np.eye(2), check_reciprocal=False),
    )
    m.add_flow(TransitionFlow("inf", state["S"], state["I"], foi))
    return m.compile()


y_age = np.array([100.0, 200.0, 10.0, 20.0, 0.0, 0.0])
built_in = np.asarray(build(FOIKind.FREQUENCY).observe(0.0, y_age, {}).captures["infection"].data)
custom = np.asarray(build(as_frequency).observe(0.0, y_age, {}).captures["infection"].data)
np.testing.assert_array_equal(built_in, custom)
print("callable kind=as_frequency matches kind=FOIKind.FREQUENCY")


A power-frequency shedding law is the same pattern — only the formula changes:


In [ ]:
def power_frequency(infectious: GroupedRate, denominator: GroupedRate) -> GroupedRate:
    # GroupedRate supports +, -, *, / today; raise via the array and re-wrap.
    ratio = infectious / denominator
    return GroupedRate(ratio.data ** 0.5, ratio.properties)


cm_power = build(power_frequency)
power_foi = np.asarray(cm_power.observe(0.0, y_age, {}).captures["infection"].data)
freq_foi = built_in
np.testing.assert_allclose(power_foi, np.sqrt(freq_foi), rtol=1e-5, atol=1e-5)
print("power-frequency FOI is sqrt of frequency FOI (contact_rate=1)")


## Appendix — name a reusable process (optional)

Only needed when you will reuse the **same** named process in many models and
want `ForceOfPredation(...)` sugar. If you need the process once, stay on rung 1 or 2.

This is the extension point {class}`~summer4.epi.ForceOfInfection` itself uses:
subclass {class}`~summer4.flows.rates.RateOps`, register an evaluator, return a
{class}`~summer4.GroupedRate` (or compose from `Reduce`).

Your class **must** define `__rate_bytes__(self) -> bytes`, encoding every field
that changes the node's value. A compiled model's identity is a digest of its
rate expressions, and the model is a static argument to `jax.jit`: if two models
differing only in your node's fields encode to the same bytes they compare equal
and share one compiled program, so the second silently returns the first's
numbers. `register_rate_eval` raises `TypeError` when the class omits the dunder,
so the mistake surfaces at import rather than as wrong output. `ForceOfPredation`
below shows the shape: a tag for the class, then each field in a stable encoding
(`_rate_bytes` for child rates, `_selector_bytes` for selectors).


In [ ]:
from dataclasses import dataclass

from summer4.flows.rates import RateOps, as_rate, register_rate_eval
from summer4.selectors import Selector


@dataclass(frozen=True, slots=True)
class ForceOfPredation(RateOps):
    """Per-group predator count × attack rate — usable as an ExitFlow rate."""

    name: str
    predators: Selector
    group_by: Property
    attack_rate: RateOps

    def __init__(
        self,
        name: str,
        *,
        predators: Selector,
        group_by: Property,
        attack_rate: object = 1.0,
    ) -> None:
        object.__setattr__(self, "name", name)
        object.__setattr__(self, "predators", predators)
        object.__setattr__(self, "group_by", group_by)
        object.__setattr__(self, "attack_rate", as_rate(attack_rate))

    def __rate_bytes__(self) -> bytes:
        from summer4.flows.rates import _rate_bytes, _selector_bytes

        return (
            b"fop"
            + self.name.encode()
            + self.group_by.name.encode()
            + _selector_bytes(self.predators)
            + _rate_bytes(self.attack_rate)
        )


@register_rate_eval(ForceOfPredation)
def _eval_force_of_predation(
    expr: ForceOfPredation,
    *,
    eval_child,
    derived,
    pmap,
    t,
    y_arr,
    captures,
) -> object:
    predators = eval_child(Reduce(sum_over=expr.group_by, where=expr.predators))
    result = predators * eval_child(expr.attack_rate)
    captures[expr.name] = result
    return result


m_named = FlowModel(pmap_lv)
m_named.add_flow(
    EntryFlow("prey_birth", species["prey"], r * Reduce(sum_over=pop_lv, where=species["prey"]))
)
m_named.add_flow(
    ExitFlow(
        "predation",
        species["prey"],
        ForceOfPredation(
            "predation",
            predators=species["predator"],
            group_by=pop_lv,
            attack_rate=alpha,
        ),
    )
)
m_named.add_flow(
    EntryFlow("conversion", species["predator"], beta_conv * FlowRef("predation").sum())
)
m_named.add_flow(ExitFlow("pred_death", species["predator"], delta))

dy_named = np.asarray(m_named.compile().vector_field(0.0, y_lv, {}))
np.testing.assert_allclose(dy_named, expected, rtol=1e-5, atol=1e-5)
print("ForceOfPredation matches the Reduce-based Lotka–Volterra model")


## What to reach for

| Need | Use |
|---|---|
| One-off custom hazard that is arithmetic on group counts | Rung 1: `Reduce` |
| One-off custom hazard that is easier to write as code | Rung 2: `defer` |
| FOI mixing / infectiousness, different shedding | Rung 3: callable `kind=` |
| Reusable named process across models | Appendix: `RateOps` + `register_rate_eval` |
| Several related couplings | Several flows (and `FlowRef` when one mass feeds another) |
| `sin`, `cos`, or any other NumPy ufunc on a rate | `np.sin(expr)` — not `jnp.sin(expr)` |

Any NumPy ufunc now works on a rate expression, and on an evaluated
`Output`, `GroupedRate` or `PropertyData`. The spelling is `np.*`.
`jnp.sin(expr)` still raises, because JAX implements no dispatch hook.

There is no multi-output "process" object that registers several flows at once.
Compose flows; share Python variables or `FlowRef` when quantities must match.
